# Train Route Analysis and Journey Time Prediction
**Internship Project — Sysslan IT Solutions**

**Tool:** Python | Jupyter Notebook  
**Dataset:** Indian Railway Station-level data (186,074 records)  
**Model:** Linear Regression  


In [ ]:
# ============================================================
# SYSSLAN IT SOLUTIONS - DATA SCIENCE INTERNSHIP PROJECT
# Title : Train Route Analysis and Journey Time Prediction
# Author: [Your Name]
# Tool  : Python | Jupyter Notebook
# ============================================================


In [ ]:
# ── CELL 1 ── Import Required Libraries ─────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set a consistent visual style for all charts
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("=" * 50)
print("  Libraries loaded. Project environment ready.")
print("=" * 50)


In [ ]:
# ── CELL 2 ── Task 1.1: Load Dataset and Summarize ──────────
rail_data = pd.read_csv("Dataset1.csv")

total_rows, total_cols = rail_data.shape
print(f"Dataset loaded successfully!")
print(f"  Total Records  : {total_rows:,}")
print(f"  Total Columns  : {total_cols}")
print(f"\nAvailable Columns:\n  {list(rail_data.columns)}")
print(f"\nFirst look at data:")
rail_data.head(8)


In [ ]:
# ── CELL 3 ── Task 1.3 & 1.4: Statistics and Missing Values ─
print("─── Basic Statistical Summary ───")
print(rail_data.describe())

print("\n─── Missing / Null Values Per Column ───")
null_summary = rail_data.isnull().sum().reset_index()
null_summary.columns = ['Column', 'Missing_Count']
null_summary['Missing_%'] = (null_summary['Missing_Count'] / total_rows * 100).round(2)
print(null_summary.to_string(index=False))

print(f"\n─── Duplicate Rows : {rail_data.duplicated().sum()} ───")


In [ ]:
# ── CELL 4 ── Task 2.1 & 2.2: Clean Data & Fix Time Format ──
# Remove duplicates
rail_data.drop_duplicates(inplace=True)

# Convert time columns to datetime format
rail_data['Arrival_time']    = pd.to_datetime(rail_data['Arrival_time'],
                                               format='%H:%M:%S', errors='coerce')
rail_data['Departure_Time']  = pd.to_datetime(rail_data['Departure_Time'],
                                               format='%H:%M:%S', errors='coerce')

# Convert Distance column to numeric
rail_data['Distance'] = pd.to_numeric(rail_data['Distance'], errors='coerce')

# Drop rows where essential columns are empty
rail_data.dropna(subset=['Arrival_time', 'Departure_Time', 'Distance'], inplace=True)

print("Data cleaning complete.")
print(f"  Rows remaining after cleaning : {len(rail_data):,}")


In [ ]:
# ── CELL 5 ── Task 2.3 & 2.4: Journey Duration + Features ───
# Sort by train and stop number
rail_data.sort_values(['Train_No', 'SN'], inplace=True)
rail_data.reset_index(drop=True, inplace=True)

# For each train, calculate duration from the first departure
def compute_elapsed_time(group):
    origin_time = group['Departure_Time'].iloc[0]
    group['Elapsed_Minutes'] = (
        group['Arrival_time'] - origin_time
    ).dt.total_seconds() / 60
    return group

rail_data = rail_data.groupby('Train_No', group_keys=False).apply(compute_elapsed_time)

# Keep only valid (positive) durations
rail_data = rail_data[rail_data['Elapsed_Minutes'] >= 0].copy()

# Feature: number of stops per train
stop_counts = (
    rail_data.groupby('Train_No')['SN']
    .max()
    .reset_index()
    .rename(columns={'SN': 'Num_Stops'})
)
rail_data = rail_data.merge(stop_counts, on='Train_No', how='left')

print("Feature engineering complete!")
print(f"  Rows with valid journey duration : {len(rail_data):,}")
rail_data[['Train_No', 'Station_Name', 'Distance', 'Num_Stops', 'Elapsed_Minutes']].head(10)


In [ ]:
# ── CELL 6 ── Task 1.2: Train-wise Route Table ───────────────
route_summary = (
    rail_data.groupby('Train_No')
    .apply(lambda g: pd.Series({
        'Origin_Station'  : g.loc[g['SN'].idxmin(), 'Station_Name'],
        'Destination'     : g.loc[g['SN'].idxmax(), 'Station_Name'],
        'Total_Stops'     : int(g['Num_Stops'].max()),
        'Max_Distance_km' : g['Distance'].max()
    }))
    .reset_index()
)

print(f"Total unique trains in dataset : {len(route_summary)}")
print("\nSample Train Route Summary (first 15 trains):")
route_summary.head(15)


In [ ]:
# ── CELL 7 ── Task 3.1 & 4.1: Journey Duration by Route ─────
route_duration = (
    rail_data.groupby('Train_No')['Elapsed_Minutes']
    .max()
    .sort_values(ascending=False)
    .head(12)
    .reset_index()
)
route_duration.columns = ['Train_No', 'Max_Duration_Min']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(
    route_duration['Train_No'].astype(str),
    route_duration['Max_Duration_Min'],
    color=plt.cm.viridis(np.linspace(0.3, 0.85, len(route_duration)))
)
ax.set_xlabel('Journey Duration (Minutes)')
ax.set_ylabel('Train Number')
ax.set_title('Top 12 Trains by Longest Journey Duration')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x)} min'))
for bar, val in zip(bars, route_duration['Max_Duration_Min']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('chart_journey_duration.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── CELL 8 ── Task 3.2 & 4.2: Station-wise Train Traffic ────
traffic_by_station = (
    rail_data.groupby('Station_Name')['Train_No']
    .nunique()
    .sort_values(ascending=False)
)

print("── TOP 10 BUSIEST STATIONS ──")
print(traffic_by_station.head(10).to_string())
print("\n── 10 LEAST BUSY STATIONS ──")
print(traffic_by_station.tail(10).to_string())

# Chart: top 15 stations
top_stations = traffic_by_station.head(15)
fig, ax = plt.subplots(figsize=(13, 5))
colors = sns.color_palette("rocket", len(top_stations))
ax.bar(top_stations.index, top_stations.values, color=colors, edgecolor='white', linewidth=0.6)
ax.set_title('Top 15 Stations by Train Traffic Volume')
ax.set_xlabel('Station Name')
ax.set_ylabel('Number of Trains Passing')
plt.xticks(rotation=50, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('chart_station_traffic.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── CELL 9 ── Task 3.3 & 4.3: Distance vs Duration Plot ─────
sample_df = rail_data.sample(n=min(8000, len(rail_data)), random_state=7)

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    sample_df['Distance'],
    sample_df['Elapsed_Minutes'],
    alpha=0.25, s=6,
    c=sample_df['Num_Stops'],
    cmap='plasma'
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Number of Stops', fontsize=10)
ax.set_title('Relationship Between Distance and Journey Duration\n(colour = number of stops)')
ax.set_xlabel('Distance from Origin (km)')
ax.set_ylabel('Journey Duration (Minutes)')
plt.tight_layout()
plt.savefig('chart_distance_vs_duration.png', bbox_inches='tight')
plt.show()

# Correlation value
corr_val = rail_data[['Distance', 'Elapsed_Minutes']].corr().iloc[0, 1]
print(f"Pearson Correlation (Distance ↔ Duration) : {corr_val:.4f}")


In [ ]:
# ── CELL 10 ── Task 4.4: Correlation Heatmap ─────────────────
numeric_cols = ['Distance', 'Num_Stops', 'Elapsed_Minutes', '1A', '2A', '3A', 'SL']
heatmap_df = rail_data[numeric_cols].apply(pd.to_numeric, errors='coerce').dropna()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(heatmap_df.corr(), dtype=bool))
sns.heatmap(
    heatmap_df.corr(), annot=True, fmt='.2f',
    cmap='coolwarm', mask=mask,
    linewidths=0.5, ax=ax,
    annot_kws={"size": 10}
)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('chart_correlation_heatmap.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── CELL 11 ── Task 5.1 & 5.2: Prepare Modeling Dataset ─────
model_features = ['Distance', 'Num_Stops']
target_col     = 'Elapsed_Minutes'

modeling_df = (
    rail_data[model_features + [target_col]]
    .apply(pd.to_numeric, errors='coerce')
    .dropna()
)

X_features = modeling_df[model_features]
y_target   = modeling_df[target_col]

# 80% training, 20% testing split
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_target,
    test_size=0.20,
    random_state=99
)

print(f"Modeling dataset ready.")
print(f"  Training samples : {len(X_train):,}")
print(f"  Testing samples  : {len(X_test):,}")
print(f"  Features used    : {model_features}")


In [ ]:
# ── CELL 12 ── Task 5.3: Build Linear Regression Model ──────
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print("Linear Regression model trained successfully!")
print(f"\nModel Coefficients:")
for feat, coef in zip(model_features, lr_model.coef_):
    print(f"  {feat:15s} → {coef:.4f}")
print(f"  Intercept        → {lr_model.intercept_:.4f}")


In [ ]:
# ── CELL 13 ── Task 5.4: Evaluate Model (MAE, RMSE, R²) ─────
y_predicted = lr_model.predict(X_test)

mae_score  = mean_absolute_error(y_test, y_predicted)
rmse_score = np.sqrt(mean_squared_error(y_test, y_predicted))
r2         = r2_score(y_test, y_predicted)

print("=" * 45)
print("       MODEL PERFORMANCE REPORT")
print("=" * 45)
print(f"  R² Score  : {r2:.4f}   (1.0 = perfect)")
print(f"  MAE       : {mae_score:.2f} minutes")
print(f"  RMSE      : {rmse_score:.2f} minutes")
print("=" * 45)
print(f"\n  Interpretation:")
print(f"  → The model explains {r2*100:.1f}% of variance in journey time.")
print(f"  → On average, predictions differ by {mae_score:.1f} minutes from actual.")
print(f"  → RMSE of {rmse_score:.1f} min penalises larger prediction errors more.")


In [ ]:
# ── CELL 14 ── Task 6.1: Actual vs Predicted Visualization ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Actual vs Predicted scatter
axes[0].scatter(y_test, y_predicted, alpha=0.2, s=6, color='#2E86AB')
axes[0].plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color='#E84855', linestyle='--', linewidth=2, label='Perfect Prediction'
)
axes[0].set_title('Actual vs Predicted Journey Duration')
axes[0].set_xlabel('Actual Duration (min)')
axes[0].set_ylabel('Predicted Duration (min)')
axes[0].legend()

# Plot 2: Residuals distribution
residuals = y_test - y_predicted
axes[1].hist(residuals, bins=60, color='#3BB273', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='#E84855', linestyle='--', linewidth=2)
axes[1].set_title('Residuals Distribution (Prediction Errors)')
axes[1].set_xlabel('Error (Actual − Predicted) in Minutes')
axes[1].set_ylabel('Frequency')

plt.suptitle('Linear Regression Model Evaluation', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('chart_model_evaluation.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── CELL 15 ── Live Prediction Demo ──────────────────────────
print("─── Journey Time Predictor ───\n")

test_cases = [
    {'Distance': 300,  'Num_Stops': 8},
    {'Distance': 750,  'Num_Stops': 15},
    {'Distance': 1200, 'Num_Stops': 22},
]

for case in test_cases:
    input_df = pd.DataFrame([case])
    pred_min = lr_model.predict(input_df)[0]
    pred_hr  = pred_min / 60
    print(f"  Distance: {case['Distance']:>5} km | "
          f"Stops: {case['Num_Stops']:>2} | "
          f"Predicted Duration: {pred_min:.1f} min ({pred_hr:.1f} hrs)")


## Key Insights — Train Route Analysis Project

### Data Overview
- The dataset contains **186,074 station-level records** across **12 columns**
- Trains span a wide range of distances from short suburban routes to long cross-country journeys

### Patterns Discovered
- **Distance and journey duration** show a strong positive correlation —
  the farther the train travels, the longer the journey takes
- **Number of stops** is also a significant factor — more stops increase total time
- Some major junction stations (like MAO, SBC) serve a very high number of trains,
  making them key hubs in the Indian railway network

### Model Performance
- The **Linear Regression model** successfully learns the relationship between
  distance, stops, and journey duration
- MAE shows the average prediction error in minutes
- RMSE highlights that large outliers (overnight trains, special routes)
  are harder to predict precisely

### Conclusion
This project demonstrates a complete data science workflow:
from raw data loading → cleaning → exploration → visualization → machine learning.
The journey time prediction system can be used to estimate travel time
for any train route given its distance and number of stops.


In [ ]:
print("Project complete! All 6 levels and tasks finished successfully.")